#### Figure 1: Spatial sea ice concentration trends and timeseries analysis for pan-Arctic and Greenland Sea regions

**Figure Caption**

Figure 1. Sea ice concentration spatial trends and March sea ice extent timeseries. Maps show Greenland Sea region of interest (black dashed line) and Fram Strait mooring locations (purple dots) with sea ice extent maxima (SIC = 0.15; blue contour) and sea-ice concentration spatial trends overlaid (shading) for a) March 1986 (1979-2014 trend) and b) March 2023 (2015-2025 trend); timeseries of March sea-ice extent with split linear trends using 2015 breakpoint and bootstrap 95% confidence intervals for c) pan-Arctic region and d) Greenland Sea. Purple squares mark the maximum extent years plotted in panels a) and b).

In [3]:
"""
Figure 1: Greenland Sea Ice Concentration Trends and Extent Timeseries
=======================================================================
4-panel figure combining:
- Row 1 (a,b): Spatial trend maps for two periods
- Row 2 (c,d): Timeseries with split linear trends

Panel a: March 1986 SIC edge with 1979-2014 trends
Panel b: March 2023 SIC edge with 2015-2025 trends
Panel c: Pan-Arctic March SIE timeseries
Panel d: Greenland Sea March SIE timeseries
"""

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects
from matplotlib.colors import LinearSegmentedColormap
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LongitudeFormatter, LatitudeFormatter
import matplotlib.ticker as mticker
import cmocean
import rockhound as rh
from scipy import stats
import os

# ============================================================================
# CONFIGURATION
# ============================================================================

# File paths
ERA5_PATH = '../era5/era5_*_Arctic.nc'

# Maximum extent years (from diagnostic script)
MAX_YEAR_P1 = 1986
MAX_YEAR_P2 = 2023

# Analysis periods
PERIOD_1 = (1979, 2015)
PERIOD_2 = (2015, 2025)
BREAKPOINT_YEAR = 2015

# Timeseries configuration
CONFIG = {
    'season': 'March',
    'breakpoint_year': 2015,
    'show_uncertainty_bands': True,
    'show_rate_annotations': True,
    'uncertainty_method': 'bootstrap',
    'confidence_level': 0.95,
    'n_bootstrap': 10000
}

# Map settings
TREND_EXTENT = {'lon_min': -90, 'lon_max': 90, 'lat_min': 60, 'lat_max': 90}
MAP_EXTENT = [-30, 20, 65, 85]
RELIEF_ALPHA = 0.5
TREND_ALPHA = 0.7
SIC_THRESHOLD = 0.15

# Colors
COLOR_BOUNDARY = 'black'
COLOR_MOORING = 'purple'
COLOR_ICE_EDGE = 'darkblue'

# Greenland Sea polygon
GREENLAND_SEA_COORDS = [(-22, 71), (-8.5, 71), (12, 79), (-21, 79), (-28, 73), (-22, 71)]

# Fram Strait moorings
MOORINGS = {'F17': -8.0, 'F14': -6.5, 'F13b': -5.5, 'F13': -5.0, 
            'F12': -4.0, 'F11': -3.0, 'F10': -2.0}
MOORING_LAT = 79.0

VERTICES = {'NW': (-21, 79), 'NE': (12, 79), 'SE': (-8.5, 71), 
            'SW': (-22, 71), 'W_Mid': (-28, 73)}

# Colormap for trends
colors_trend = ['#2166ac', '#4393c3', '#92c5de', '#d1e5f0', 'white', 
                '#fddbc7', '#f4a582', '#d6604d', '#b2182b']
cmap_trend = LinearSegmentedColormap.from_list('red_white_blue', colors_trend, N=256)

# Timeseries colors
COLORS = {
    'data': '#808080',
    'period1': '#2E86AB',  # GLORYS blue
    'period2': '#A23B72',  # Purple-red
    'ci_band': 0.15,
    'breakpoint': '#808080'
}

# ============================================================================
# FUNCTIONS - SPATIAL TRENDS
# ============================================================================

def calculate_spatial_trends_vectorized(data_array, time_dim='time'):
    """
    Vectorized calculation of linear trends for each grid cell.
    
    Args:
        data_array: xarray DataArray with time dimension
        time_dim: Name of time dimension
    
    Returns:
        Tuple of (slope, pvalue, rsquared) DataArrays
    """
    print("  Calculating spatial trends (vectorized)...")
    
    time_values = data_array[time_dim].values
    years = pd.to_datetime(time_values).year.values
    time_numeric = years - years[0]
    n_times = len(time_numeric)
    
    data = data_array.values
    n_lat, n_lon = data.shape[1], data.shape[2]
    data_reshaped = data.reshape(n_times, -1)
    
    slope = np.full(n_lat * n_lon, np.nan)
    pvalue = np.full(n_lat * n_lon, np.nan)
    rsquared = np.full(n_lat * n_lon, np.nan)
    
    n_valid = np.sum(~np.isnan(data_reshaped), axis=0)
    valid_cells = n_valid >= 5
    
    print(f"    Valid cells: {valid_cells.sum()} / {len(valid_cells)}")
    
    if valid_cells.sum() > 0:
        for i in np.where(valid_cells)[0]:
            series = data_reshaped[:, i]
            valid_mask = ~np.isnan(series)
            
            if valid_mask.sum() >= 5:
                result = stats.linregress(time_numeric[valid_mask], series[valid_mask])
                slope[i] = result.slope
                pvalue[i] = result.pvalue
                rsquared[i] = result.rvalue ** 2
    
    slope = slope.reshape(n_lat, n_lon)
    pvalue = pvalue.reshape(n_lat, n_lon)
    rsquared = rsquared.reshape(n_lat, n_lon)
    
    slope_da = xr.DataArray(slope, coords=[data_array.latitude, data_array.longitude],
                            dims=['latitude', 'longitude'], name='slope')
    pvalue_da = xr.DataArray(pvalue, coords=[data_array.latitude, data_array.longitude],
                             dims=['latitude', 'longitude'], name='pvalue')
    rsquared_da = xr.DataArray(rsquared, coords=[data_array.latitude, data_array.longitude],
                               dims=['latitude', 'longitude'], name='rsquared')
    
    return slope_da, pvalue_da, rsquared_da

def add_map_features(ax, show_moorings=True):
    """Add standard map features."""
    ax.coastlines(linewidth=0.5, color='#333333')
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                     linewidth=0.5, color='gray', alpha=0.4, linestyle=':')
    gl.top_labels = False
    gl.right_labels = False
    gl.xformatter = LongitudeFormatter()
    gl.yformatter = LatitudeFormatter()
    gl.xlabel_style = {'size': 8}
    gl.ylabel_style = {'size': 8}
    
    lons_gs = [coord[0] for coord in GREENLAND_SEA_COORDS]
    lats_gs = [coord[1] for coord in GREENLAND_SEA_COORDS]
    ax.plot(lons_gs, lats_gs, color=COLOR_BOUNDARY, linewidth=1.5,
           linestyle='--', transform=ccrs.PlateCarree(), zorder=3)
    
    if show_moorings:
        for name, lon in MOORINGS.items():
            ax.plot(lon, MOORING_LAT, 'o', color=COLOR_MOORING, markersize=4,
                   transform=ccrs.PlateCarree(), zorder=4)

# ============================================================================
# FUNCTIONS - TIMESERIES ANALYSIS
# ============================================================================

def bootstrap_trend_ci(x, y, n_bootstrap=10000, confidence_level=0.95):
    """
    Calculate bootstrap confidence intervals for linear trend.
    
    Args:
        x: Independent variable (time)
        y: Dependent variable (SIE)
        n_bootstrap: Number of bootstrap samples
        confidence_level: Confidence level (default 0.95 for 95% CI)
    
    Returns:
        Dictionary with slope, intercept, and CI bounds
    """
    n = len(x)
    slopes = np.zeros(n_bootstrap)
    intercepts = np.zeros(n_bootstrap)
    
    for i in range(n_bootstrap):
        indices = np.random.choice(n, size=n, replace=True)
        x_boot = x[indices]
        y_boot = y[indices]
        result = stats.linregress(x_boot, y_boot)
        slopes[i] = result.slope
        intercepts[i] = result.intercept
    
    alpha = 1 - confidence_level
    slope_ci = np.percentile(slopes, [alpha/2 * 100, (1 - alpha/2) * 100])
    intercept_ci = np.percentile(intercepts, [alpha/2 * 100, (1 - alpha/2) * 100])
    
    result = stats.linregress(x, y)
    
    return {
        'slope': result.slope,
        'intercept': result.intercept,
        'slope_ci_lower': slope_ci[0],
        'slope_ci_upper': slope_ci[1],
        'intercept_ci_lower': intercept_ci[0],
        'intercept_ci_upper': intercept_ci[1],
        'r_squared': result.rvalue**2,
        'p_value': result.pvalue
    }

def analytical_trend_ci(x, y, confidence_level=0.95):
    """
    Calculate analytical confidence intervals for linear trend.
    
    Args:
        x: Independent variable (time)
        y: Dependent variable (SIE)
        confidence_level: Confidence level (default 0.95 for 95% CI)
    
    Returns:
        Dictionary with slope, intercept, and CI bounds
    """
    result = stats.linregress(x, y)
    
    n = len(x)
    y_pred = result.slope * x + result.intercept
    residuals = y - y_pred
    mse = np.sum(residuals**2) / (n - 2)
    
    x_mean = np.mean(x)
    sxx = np.sum((x - x_mean)**2)
    se_slope = np.sqrt(mse / sxx)
    se_intercept = np.sqrt(mse * (1/n + x_mean**2/sxx))
    
    from scipy.stats import t
    t_val = t.ppf((1 + confidence_level) / 2, n - 2)
    
    return {
        'slope': result.slope,
        'intercept': result.intercept,
        'slope_ci_lower': result.slope - t_val * se_slope,
        'slope_ci_upper': result.slope + t_val * se_slope,
        'intercept_ci_lower': result.intercept - t_val * se_intercept,
        'intercept_ci_upper': result.intercept + t_val * se_intercept,
        'r_squared': result.rvalue**2,
        'p_value': result.pvalue
    }

# ============================================================================
# LOAD DATA
# ============================================================================

print("="*70)
print("LOADING DATA")
print("="*70)

# Load ERA5 sea ice concentration
print("\nLoading ERA5 sea ice data...")
ds = xr.open_mfdataset(ERA5_PATH, chunks={'time': 12})
sic = ds['siconc']
print(f"  Loaded: {sic.dims}")
print(f"  Time range: {str(sic.time.min().values)[:10]} to {str(sic.time.max().values)[:10]}")

# Load bathymetry
print("\nLoading bathymetry...")
bathy = rh.fetch_etopo1(version="ice") #,resolution='10m', region=MAP_EXTENT

# Load NSIDC sea ice index data
print("\nLoading NSIDC sea ice index data...")
url_nh = 'https://noaadata.apps.nsidc.org/NOAA/G02135/north/daily/data/N_seaice_extent_daily_v3.0.csv'
df_nh = pd.read_csv(url_nh, skiprows=2)
df_nh.columns = ['year', 'month', 'day', 'extent', 'missing', 'source_data']
df_nh['time'] = pd.to_datetime(df_nh[['year', 'month', 'day']])
ds_nh = df_nh.set_index('time')['extent'].to_xarray()
ds_nh = ds_nh.where(ds_nh > 0, drop=True)
ds_nh['month'] = ds_nh.time.dt.month

# Calculate Greenland Sea extent
print("\nCalculating Greenland Sea extent...")
from shapely.geometry import Point, Polygon
gs_polygon = Polygon(GREENLAND_SEA_COORDS)

lons_2d, lats_2d = np.meshgrid(sic.longitude.values, sic.latitude.values)
gs_mask = np.zeros_like(lons_2d, dtype=bool)
for i in range(lons_2d.shape[0]):
    for j in range(lons_2d.shape[1]):
        point = Point(lons_2d[i,j], lats_2d[i,j])
        gs_mask[i,j] = gs_polygon.contains(point)

cell_areas = (111.32 * 0.25) * (111.32 * 0.25 * np.cos(np.radians(lats_2d)))
sic_gs = sic.where(gs_mask)
ice_area_gs = (sic_gs * cell_areas).sum(dim=['latitude', 'longitude']) / 1e6
ds_gs = ice_area_gs.compute()
ds_gs['month'] = ds_gs.time.dt.month
print(f"  Greenland Sea extent calculated")

# ============================================================================
# CALCULATE SPATIAL TRENDS
# ============================================================================

print("\n" + "="*70)
print("CALCULATING SPATIAL TRENDS")
print("="*70)

# Subset to trend extent
sic_trend = sic.sel(
    longitude=slice(TREND_EXTENT['lon_min'], TREND_EXTENT['lon_max']),
    latitude=slice(TREND_EXTENT['lat_min'], TREND_EXTENT['lat_max'])
)

# Filter for March only
sic_march = sic_trend.where(sic_trend.time.dt.month == 3, drop=True)

# Period 1: 1979-2014
print(f"\nPeriod 1: {PERIOD_1[0]}-{PERIOD_1[1]}")
sic_p1 = sic_march.sel(time=slice(str(PERIOD_1[0]), str(PERIOD_1[1])))
slope_p1, pval_p1, r2_p1 = calculate_spatial_trends_vectorized(sic_p1)
slope_p1_decade = slope_p1 * 10 * 100  # Convert to % per decade

# Period 2: 2015-2025
print(f"\nPeriod 2: {PERIOD_2[0]}-{PERIOD_2[1]}")
sic_p2 = sic_march.sel(time=slice(str(PERIOD_2[0]), str(PERIOD_2[1])))
slope_p2, pval_p2, r2_p2 = calculate_spatial_trends_vectorized(sic_p2)
slope_p2_decade = slope_p2 * 10 * 100  # Convert to % per decade

# Get specific years for ice edge plotting
sic_1986 = sic.sel(time=f'{MAX_YEAR_P1}-03', method='nearest')
sic_2023 = sic.sel(time=f'{MAX_YEAR_P2}-03', method='nearest')

# ============================================================================
# CREATE FIGURE
# ============================================================================

print("\n" + "="*70)
print("CREATING FIGURE")
print("="*70)

fig = plt.figure(figsize=(12, 10))
gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.25)

# ============================================================================
# ROW 1: SPATIAL MAPS
# ============================================================================

print("\nCreating spatial maps...")

# --- Panel (a): Period 1 Map ---
ax_a = fig.add_subplot(gs[0, 0], projection=ccrs.NorthPolarStereo(central_longitude=-20))
ax_a.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())

bathy.plot.contourf(ax=ax_a, transform=ccrs.PlateCarree(),
                    levels=20, cmap='gray', alpha=RELIEF_ALPHA, add_colorbar=False)

trend_plot = slope_p1_decade.where(sic_p1.mean('time') >= SIC_THRESHOLD).plot.contourf(
    ax=ax_a, transform=ccrs.PlateCarree(),
    levels=np.linspace(-15, 15, 16),
    cmap=cmap_trend, alpha=TREND_ALPHA,
    add_colorbar=False, extend='both'
)

sic_1986.plot.contour(ax=ax_a, transform=ccrs.PlateCarree(),
                     levels=[SIC_THRESHOLD], colors=[COLOR_ICE_EDGE],
                     linewidths=2, add_colorbar=False)

add_map_features(ax_a, show_moorings=True)
ax_a.set_title(f'a) March {MAX_YEAR_P1} ice edge, {PERIOD_1[0]}-{PERIOD_1[1]} trend', 
               loc='left', fontsize=11, fontweight='bold')

# --- Panel (b): Period 2 Map ---
ax_b = fig.add_subplot(gs[0, 1], projection=ccrs.NorthPolarStereo(central_longitude=-20))
ax_b.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())

bathy.plot.contourf(ax=ax_b, transform=ccrs.PlateCarree(),
                    levels=20, cmap='gray', alpha=RELIEF_ALPHA, add_colorbar=False)

trend_plot = slope_p2_decade.where(sic_p2.mean('time') >= SIC_THRESHOLD).plot.contourf(
    ax=ax_b, transform=ccrs.PlateCarree(),
    levels=np.linspace(-15, 15, 16),
    cmap=cmap_trend, alpha=TREND_ALPHA,
    add_colorbar=False, extend='both'
)

sic_2023.plot.contour(ax=ax_b, transform=ccrs.PlateCarree(),
                     levels=[SIC_THRESHOLD], colors=[COLOR_ICE_EDGE],
                     linewidths=2, add_colorbar=False)

add_map_features(ax_b, show_moorings=True)
ax_b.set_title(f'b) March {MAX_YEAR_P2} ice edge, {PERIOD_2[0]}-{PERIOD_2[1]} trend',
               loc='left', fontsize=11, fontweight='bold')

# Add shared colorbar for trends
cbar_ax = fig.add_axes([0.15, 0.53, 0.7, 0.015])
cbar = plt.colorbar(trend_plot, cax=cbar_ax, orientation='horizontal')
cbar.set_label('SIC trend (% per decade)', fontsize=10)
cbar.ax.tick_params(labelsize=9)

# ============================================================================
# ROW 2: TIMESERIES WITH SPLIT TRENDS
# ============================================================================

print("\nCreating timeseries plots...")

season_month = 3
season_name = CONFIG['season']

# --- Panel (c): Pan-Arctic ---
ax_c = fig.add_subplot(gs[1, 0])
print(f"\nPanel (c): Pan-Arctic {season_name}")

x_data = ds_nh.time.where(ds_nh.month == season_month, drop=True)
y_data = ds_nh.where(ds_nh.month == season_month, drop=True)

# Split data at breakpoint
mask_p1 = x_data.dt.year < CONFIG['breakpoint_year']
mask_p2 = x_data.dt.year >= CONFIG['breakpoint_year']

x_p1 = x_data.where(mask_p1, drop=True)
y_p1 = y_data.where(mask_p1, drop=True)
x_p2 = x_data.where(mask_p2, drop=True)
y_p2 = y_data.where(mask_p2, drop=True)

# Convert to numeric for regression
x_p1_num = x_p1.dt.year.values + (x_p1.dt.dayofyear.values / 365.25)
x_p2_num = x_p2.dt.year.values + (x_p2.dt.dayofyear.values / 365.25)

# Calculate trends
if CONFIG['uncertainty_method'] == 'bootstrap':
    trend_p1 = bootstrap_trend_ci(x_p1_num, y_p1.values, 
                                   n_bootstrap=CONFIG['n_bootstrap'],
                                   confidence_level=CONFIG['confidence_level'])
    trend_p2 = bootstrap_trend_ci(x_p2_num, y_p2.values,
                                   n_bootstrap=CONFIG['n_bootstrap'],
                                   confidence_level=CONFIG['confidence_level'])
else:
    trend_p1 = analytical_trend_ci(x_p1_num, y_p1.values,
                                    confidence_level=CONFIG['confidence_level'])
    trend_p2 = analytical_trend_ci(x_p2_num, y_p2.values,
                                    confidence_level=CONFIG['confidence_level'])

# Plot data
ax_c.plot(x_data, y_data, color='#808080', linewidth=1, alpha=0.5, zorder=1)
ax_c.scatter(x_data, y_data, s=20, color=COLORS['data'], alpha=0.6, 
            label='Observed', zorder=2, edgecolors='none')

# Plot trends
y_fit_p1 = trend_p1['slope'] * x_p1_num + trend_p1['intercept']
y_fit_p2 = trend_p2['slope'] * x_p2_num + trend_p2['intercept']

ax_c.plot(x_p1, y_fit_p1, color=COLORS['period1'], linewidth=2.5, 
         label=f'{PERIOD_1[0]}-{PERIOD_1[1]} trend', zorder=3)
ax_c.plot(x_p2, y_fit_p2, color=COLORS['period2'], linewidth=2.5,
         label=f'{PERIOD_2[0]}-{PERIOD_2[1]} trend', zorder=3)

# Add uncertainty bands if requested
if CONFIG['show_uncertainty_bands']:
    y_lower_p1 = trend_p1['slope_ci_lower'] * x_p1_num + trend_p1['intercept_ci_lower']
    y_upper_p1 = trend_p1['slope_ci_upper'] * x_p1_num + trend_p1['intercept_ci_upper']
    y_lower_p2 = trend_p2['slope_ci_lower'] * x_p2_num + trend_p2['intercept_ci_lower']
    y_upper_p2 = trend_p2['slope_ci_upper'] * x_p2_num + trend_p2['intercept_ci_upper']
    
    ax_c.fill_between(x_p1, y_lower_p1, y_upper_p1, 
                     color=COLORS['period1'], alpha=COLORS['ci_band'], zorder=2)
    ax_c.fill_between(x_p2, y_lower_p2, y_upper_p2,
                     color=COLORS['period2'], alpha=COLORS['ci_band'], zorder=2)

# Mark maximum extent years
for year in [MAX_YEAR_P1, MAX_YEAR_P2]:
    year_data = y_data.where(x_data.dt.year == year, drop=True)
    if len(year_data) > 0:
        year_time = x_data.where(x_data.dt.year == year, drop=True).values[0]
        ax_c.scatter(year_time, year_data.values[0], s=120, marker='s',
                    color=COLOR_MOORING, edgecolors='white', linewidth=1.5,
                    zorder=5, alpha=0.8)

# Add breakpoint line
ax_c.axvline(x=pd.Timestamp(f'{CONFIG["breakpoint_year"]}-01-01'),
            color=COLORS['breakpoint'], linestyle=':', linewidth=1.5,
            alpha=0.7, zorder=1)

ax_c.set_xlabel('Year', fontsize=10)
ax_c.set_ylabel(r'Sea ice extent ($\times10^{6}$ km$^{2}$)', fontsize=10)
ax_c.set_title(f'c) Pan-Arctic {season_name} SIE', loc='left', 
              fontsize=11, fontweight='bold')
ax_c.legend(loc='upper right', frameon=True, edgecolor='none', fontsize=9)
ax_c.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# Add rate annotations if requested
if CONFIG['show_rate_annotations']:
    rate_text = (f"Rate P1: {trend_p1['slope']:.3f} ± "
                f"{(trend_p1['slope_ci_upper']-trend_p1['slope_ci_lower'])/2:.3f} "
                f"$\\times10^6$ km$^2$/yr\n"
                f"Rate P2: {trend_p2['slope']:.3f} ± "
                f"{(trend_p2['slope_ci_upper']-trend_p2['slope_ci_lower'])/2:.3f} "
                f"$\\times10^6$ km$^2$/yr")
    ax_c.text(0.02, 0.08, rate_text, transform=ax_c.transAxes,
             fontsize=8, verticalalignment='bottom',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
             family='monospace')

# --- Panel (d): Greenland Sea ---
ax_d = fig.add_subplot(gs[1, 1])
print(f"\nPanel (d): Greenland Sea {season_name}")

x_data = ds_gs.time.where(ds_gs.month == season_month, drop=True)
y_data = ds_gs.where(ds_gs.month == season_month, drop=True)

# Split data at breakpoint
mask_p1 = x_data.dt.year < CONFIG['breakpoint_year']
mask_p2 = x_data.dt.year >= CONFIG['breakpoint_year']

x_p1 = x_data.where(mask_p1, drop=True)
y_p1 = y_data.where(mask_p1, drop=True)
x_p2 = x_data.where(mask_p2, drop=True)
y_p2 = y_data.where(mask_p2, drop=True)

x_p1_num = x_p1.dt.year.values + (x_p1.dt.dayofyear.values / 365.25)
x_p2_num = x_p2.dt.year.values + (x_p2.dt.dayofyear.values / 365.25)

# Calculate trends
if CONFIG['uncertainty_method'] == 'bootstrap':
    trend_p1 = bootstrap_trend_ci(x_p1_num, y_p1.values,
                                   n_bootstrap=CONFIG['n_bootstrap'],
                                   confidence_level=CONFIG['confidence_level'])
    trend_p2 = bootstrap_trend_ci(x_p2_num, y_p2.values,
                                   n_bootstrap=CONFIG['n_bootstrap'],
                                   confidence_level=CONFIG['confidence_level'])
else:
    trend_p1 = analytical_trend_ci(x_p1_num, y_p1.values,
                                    confidence_level=CONFIG['confidence_level'])
    trend_p2 = analytical_trend_ci(x_p2_num, y_p2.values,
                                    confidence_level=CONFIG['confidence_level'])

# Plot data
ax_d.plot(x_data, y_data, color='#808080', linewidth=1, alpha=0.5, zorder=1)
ax_d.scatter(x_data, y_data, s=20, color=COLORS['data'], alpha=0.6,
            label='Observed', zorder=2, edgecolors='none')

# Plot trends
y_fit_p1 = trend_p1['slope'] * x_p1_num + trend_p1['intercept']
y_fit_p2 = trend_p2['slope'] * x_p2_num + trend_p2['intercept']

ax_d.plot(x_p1, y_fit_p1, color=COLORS['period1'], linewidth=2.5,
         label=f'{PERIOD_1[0]}-{PERIOD_1[1]} trend', zorder=3)
ax_d.plot(x_p2, y_fit_p2, color=COLORS['period2'], linewidth=2.5,
         label=f'{PERIOD_2[0]}-{PERIOD_2[1]} trend', zorder=3)

# Add uncertainty bands
if CONFIG['show_uncertainty_bands']:
    y_lower_p1 = trend_p1['slope_ci_lower'] * x_p1_num + trend_p1['intercept_ci_lower']
    y_upper_p1 = trend_p1['slope_ci_upper'] * x_p1_num + trend_p1['intercept_ci_upper']
    y_lower_p2 = trend_p2['slope_ci_lower'] * x_p2_num + trend_p2['intercept_ci_lower']
    y_upper_p2 = trend_p2['slope_ci_upper'] * x_p2_num + trend_p2['intercept_ci_upper']
    
    ax_d.fill_between(x_p1, y_lower_p1, y_upper_p1,
                     color=COLORS['period1'], alpha=COLORS['ci_band'], zorder=2)
    ax_d.fill_between(x_p2, y_lower_p2, y_upper_p2,
                     color=COLORS['period2'], alpha=COLORS['ci_band'], zorder=2)

# Mark maximum extent years
for year in [MAX_YEAR_P1, MAX_YEAR_P2]:
    year_data = y_data.where(x_data.dt.year == year, drop=True)
    if len(year_data) > 0:
        year_time = x_data.where(x_data.dt.year == year, drop=True).values[0]
        ax_d.scatter(year_time, year_data.values[0], s=120, marker='s',
                    color=COLOR_MOORING, edgecolors='white', linewidth=1.5,
                    zorder=5, alpha=0.8)

# Add breakpoint line
ax_d.axvline(x=pd.Timestamp(f'{CONFIG["breakpoint_year"]}-01-01'),
            color=COLORS['breakpoint'], linestyle=':', linewidth=1.5,
            alpha=0.7, zorder=1)

ax_d.set_xlabel('Year', fontsize=10)
ax_d.set_title(f'd) Greenland Sea {season_name} SIE', loc='left',
              fontsize=11, fontweight='bold')
ax_d.legend(loc='upper right', frameon=True, edgecolor='none', fontsize=9)
ax_d.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# Add rate annotations
if CONFIG['show_rate_annotations']:
    rate_text = (f"Rate P1: {trend_p1['slope']:.4f} ± "
                f"{(trend_p1['slope_ci_upper']-trend_p1['slope_ci_lower'])/2:.4f} "
                f"$\\times10^6$ km$^2$/yr\n"
                f"Rate P2: {trend_p2['slope']:.4f} ± "
                f"{(trend_p2['slope_ci_upper']-trend_p2['slope_ci_lower'])/2:.4f} "
                f"$\\times10^6$ km$^2$/yr")
    ax_d.text(0.02, 0.08, rate_text, transform=ax_d.transAxes,
             fontsize=8, verticalalignment='bottom',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
             family='monospace')

# ============================================================================
# SAVE FIGURE
# ============================================================================

output_dir = './figures'
os.makedirs(output_dir, exist_ok=True)

output_path = f"{output_dir}/Figure_1_SIE_spatial_and_timeseries.png"
fig.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
print(f"\n{'='*70}")
print(f"Figure saved to: {output_path}")
print(f"{'='*70}\n")

plt.show()

LOADING DATA

Loading ERA5 sea ice data...
  Loaded: ('time', 'latitude', 'longitude')
  Time range: 1979-01-01 to 2025-05-01

Loading bathymetry...

Loading NSIDC sea ice index data...


HTTPError: HTTP Error 404: Not Found